# Reconnaissance vocale (ASR) en éwé — Fine-tuning de MMS (facebook/mms-1b-all)

**ASR** = *Automatic Speech Recognition* : transformer un **son** (la voix) en **texte**.
Contrairement à Whisper, on utilise ici **MMS** (*Massively Multilingual Speech*) de Meta,
qui prend en charge l'**éwé nativement** (parmi >1000 langues). C'est souvent **meilleur**
que Whisper pour les langues africaines peu dotées.

## MMS, comment ça marche ? (CTC vs encodeur-décodeur)

MMS repose sur **wav2vec2** et la perte **CTC** (*Connectionist Temporal Classification*).
La grande différence avec Whisper :

| | Whisper | MMS (wav2vec2-CTC) |
|---|---|---|
| Architecture | encodeur → **décodeur** (génératif) | encodeur **seul** + tête CTC |
| Entrée | spectrogramme log-Mel | **forme d'onde brute** (16 kHz) |
| Sortie | tokens de sous-mots générés | **un caractère par pas de temps**, fusionnés par CTC |
| Vocabulaire | fixe (BPE multilingue) | **construit à partir de nos transcriptions** |

**CTC en une phrase :** le modèle prédit, pour chaque petite tranche de temps, une lettre
(ou un « blanc »). CTC se charge ensuite de fusionner les répétitions et de retirer les blancs
pour reconstruire le texte — sans avoir besoin d'aligner manuellement audio et caractères.

```mermaid
flowchart LR
    A[Audio .flac 48 kHz] -->|reechantillonnage 16 kHz| B[Forme d onde brute]
    B --> C[Encodeur wav2vec2 1B]
    C --> D[Adaptateur de langue eve]
    D --> E[Tete CTC: 1 lettre par pas de temps]
    E -->|fusion CTC| F[Texte eve]
```

## L'astuce MMS : les **adaptateurs** de langue

MMS-1B contient **1 milliard** de paramètres partagés + de petits **adaptateurs** (~2 M
paramètres) spécifiques à chaque langue. On **gèle le gros modèle** et on n'entraîne que
l'adaptateur éwé : très rapide, peu de VRAM, et il existe déjà un adaptateur `ewe` de départ
qu'on va simplement affiner sur nos données.


## 0. Installation des dépendances

- `transformers`, `datasets` : modèle + données
- `evaluate`, `jiwer` : métriques **WER** / **CER**
- `librosa`, `soundfile` : lecture/rééchantillonnage audio
- `accelerate` : entraînement optimisé

In [ ]:
!pip install -q transformers datasets evaluate jiwer accelerate librosa soundfile

## 1. Imports et configuration

On force **un seul GPU** (Kaggle T4 x2) et on réduit la fragmentation mémoire **avant**
d'importer `torch`. `TARGET_LANG = "ewe"` est le code ISO 639-3 de l'éwé reconnu par MMS.

In [ ]:
import os
import re
import json

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import torch
import numpy as np
import evaluate
from datasets import load_dataset, Audio
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Configuration ----
MODEL_NAME    = "facebook/mms-1b-all"
TARGET_LANG   = "ewe"                                  # code ISO 639-3 de l'eve (adaptateur MMS)
OUTPUT_DIR    = "./output/mms-ewe-mixed"               # nouveau dossier (ne pas ecraser le v1)
DATASET_ID    = "romaricnadjire/ewe-asr-mixed-16k"     # dataset fusionne 42 h (5 sources, 16 kHz)
PUSH_REPO     = "romaricnadjire/mms-ewe-asr-mixed"     # depot de publication de l'adaptateur
SAMPLING_RATE = 16_000                                 # MMS attend du 16 kHz
MAX_AUDIO_SEC = 20.0                                   # filtrer les audios trop longs (CTC/VRAM)

# ---- Mode essai : TRIAL=True => sous-ensemble + peu de steps + PAS de push Hub (reglage rapide) ----
# TRIAL=False => run complet (tout le corpus, 8000 steps, checkpoints Hub, reprise multi-session).
TRIAL = False
MAX_TRAIN_SAMPLES = 4000 if TRIAL else None            # 4000 pour l'essai ; tout le corpus sinon

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("device :", device, "| langue cible :", TARGET_LANG, "| dataset :", DATASET_ID, "| TRIAL :", TRIAL)

In [ ]:
# ---- Authentification Hugging Face + tracker (W&B en ligne, sinon TensorBoard local) ----
import sys, subprocess
from huggingface_hub import login

def _get_secret(name):
    # Kaggle Secrets en priorite, sinon variable d'environnement (.env en local)
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

HF_TOKEN_READ  = _get_secret("HF_TOKEN_READ")  or _get_secret("HF_TOKEN")
HF_TOKEN_WRITE = _get_secret("HF_TOKEN_WRITE") or _get_secret("HF_TOKEN")
if HF_TOKEN_WRITE:
    login(token=HF_TOKEN_WRITE)
    os.environ["HF_TOKEN"] = HF_TOKEN_WRITE      # pour load_dataset(token=True)

# Tracker : W&B si la cle est presente (suivi multi-session en ligne), sinon TensorBoard.
WANDB_KEY = _get_secret("WANDB_API_KEY")
if WANDB_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_KEY
    os.environ["WANDB_PROJECT"] = "mms-ewe-asr"
    os.environ["WANDB_RESUME"]  = "allow"        # reprend le MEME run d'une session a l'autre
    # Essai isole sous un run distinct pour ne pas polluer le run principal.
    os.environ["WANDB_RUN_ID"]  = "mms-ewe-mixed-trial" if TRIAL else "mms-ewe-mixed"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=False)
    REPORT_TO = ["wandb"]
else:
    REPORT_TO = ["tensorboard"]
print("Tracker :", REPORT_TO, "| push Hub :", bool(HF_TOKEN_WRITE) and not TRIAL, "->", PUSH_REPO)

## 2. Chargement du dataset + renommage de colonne

Le dataset HF a été créé pour Whisper, donc sa colonne texte s'appelle `transcription`.
**Pas besoin de ré-uploader le dataset** : la convention wav2vec2/MMS est d'utiliser une
colonne `sentence`, alors on la **renomme simplement après chargement**.

On rééchantillonne aussi l'audio à 16 kHz (conversion automatique et paresseuse).

In [ ]:
import time

# ---- Chargement optimise : version Parquet si dispo (quelques gros fichiers -> pas de HEAD
#      par .wav, donc pas de rate-limit 429), sinon audiofolder (lent, une seule fois) PUIS
#      publication de la version Parquet pour rendre les chargements suivants rapides. ----
DATASET_PQ = DATASET_ID + "-pq"
os.environ.setdefault("HF_TOKEN", HF_TOKEN_WRITE or HF_TOKEN_READ or "")

# Le Hub renvoie parfois une 504 / timeout en listant l'arborescence d'un gros dataset
# (~37 k fichiers). On allonge les delais et on reessaie avec backoff exponentiel.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")

def _load_dataset_retry(dataset_id, tries=6, base=5):
    for k in range(tries):
        try:
            return load_dataset(dataset_id, token=True)
        except Exception as e:
            if k == tries - 1:
                raise
            wait = base * (2 ** k)
            print(f"[load_dataset] {type(e).__name__} -> nouvel essai {k + 1}/{tries - 1} dans {wait}s")
            time.sleep(wait)

def _load_audio_dataset():
    # 1) Version Parquet (rapide, aucun HEAD par fichier -> pas de 429).
    try:
        d = load_dataset(DATASET_PQ, token=True)
        if all(s in d and len(d[s]) > 0 for s in ("train", "validation", "test")):
            print(f"Charge depuis {DATASET_PQ} (Parquet, rapide).")
            return d
        print(f"{DATASET_PQ} incomplet -> audiofolder.")
    except Exception as e:
        print(f"Parquet indisponible ({type(e).__name__}) -> audiofolder (lent, une seule fois).")
    # 2) Audiofolder (37 k .wav, lent a cause du rate-limit HF).
    d = _load_dataset_retry(DATASET_ID)
    # 3) Publier la version Parquet pour les prochaines sessions (non bloquant).
    if HF_TOKEN_WRITE and not TRIAL:
        try:
            print(f"Publication Parquet -> {DATASET_PQ} (chargements suivants rapides)...")
            d.push_to_hub(DATASET_PQ, private=True, token=HF_TOKEN_WRITE)
            print("Version Parquet publiee.")
        except Exception as e:
            print(f"Push Parquet echoue (non bloquant) : {type(e).__name__}: {e}")
    return d

ds = _load_audio_dataset()

# Convention MMS / wav2vec2 : colonne texte = 'sentence'
if "transcription" in ds["train"].column_names:
    ds = ds.rename_column("transcription", "sentence")

ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

if MAX_TRAIN_SAMPLES:
    ds["train"] = ds["train"].select(range(min(MAX_TRAIN_SAMPLES, len(ds["train"]))))

print(ds)

In [ ]:
# Inspecter un exemple
ex = ds["train"][0]
print("Phrase        :", ex["sentence"])
print("Sampling rate :", ex["audio"]["sampling_rate"])
print("Duree (s)     :", round(len(ex["audio"]["array"]) / ex["audio"]["sampling_rate"], 2))

# Vue d'ensemble du dataset fusionne (5 sources)
print("\nClips par split :", {k: ds[k].num_rows for k in ds})
if "dataset" in ds["train"].column_names:
    from collections import Counter
    print("Repartition par source :", dict(Counter(ds["train"]["dataset"])))

## 3. Normalisation du texte

Pour un modèle CTC au niveau **caractère**, on nettoie d'abord la ponctuation (qui ne
s'« entend » pas) et on met en minuscules. **Important :** on **conserve** toutes les
lettres et diacritiques de l'éwé (ɖ, ɔ, ŋ, ɛ, ʋ, ƒ, accents de ton…) — ce sont des sons,
pas de la ponctuation. La mise en minuscule de Python gère correctement ces lettres
(Ɖ→ɖ, Ɔ→ɔ, Ŋ→ŋ, Ɛ→ɛ…).

In [ ]:
# Caracteres a supprimer : ponctuation et guillemets typographiques (pas les lettres eve)
chars_to_remove_regex = r'[\,\?\.\!\-\;\:\"“”„‟‘’«»…()\[\]/]'

def remove_special_characters(batch):
    text = re.sub(chars_to_remove_regex, "", batch["sentence"])
    batch["sentence"] = text.lower().strip()
    return batch

ds = ds.map(remove_special_characters, desc="Nettoyage du texte")
print("Exemple nettoye :", ds["train"][0]["sentence"])


## 4. Construction du vocabulaire de caractères

Contrairement à Whisper (vocabulaire BPE figé), un modèle CTC a besoin d'un vocabulaire
**explicite** : la liste de **tous les caractères** présents dans nos transcriptions.

On parcourt le corpus, on collecte l'ensemble des caractères, puis on ajoute :
- `|` : symbole de séparation des mots (l'espace) ;
- `[UNK]` : caractère inconnu ;
- `[PAD]` : le « blanc » CTC (token de remplissage).

**Spécificité MMS :** le vocabulaire est imbriqué sous la clé de langue
(`{"ewe": {...}}`) car le tokenizer MMS peut contenir plusieurs langues.

In [ ]:
def extract_all_chars(batch):
    all_text = " ".join(batch["sentence"])
    return {"vocab": [sorted(set(all_text))], "all_text": [all_text]}

vocabs = ds.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=ds["train"].column_names,
)

# Union des caracteres de tous les splits
all_chars = set()
for split in vocabs:
    all_chars |= set(vocabs[split]["vocab"][0])

vocab_list = sorted(all_chars)
vocab_dict = {c: i for i, c in enumerate(vocab_list)}

# Espace -> '|' (separateur de mots, plus visible)
if " " in vocab_dict:
    vocab_dict["|"] = vocab_dict.pop(" ")
# Tokens speciaux
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print(f"Taille du vocabulaire : {len(vocab_dict)} symboles")
print("Caracteres :", "".join(c for c in vocab_list if c.strip()))

# Format MMS : vocabulaire imbrique sous la cle de langue
nested_vocab = {TARGET_LANG: vocab_dict}
with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(nested_vocab, f, ensure_ascii=False)
print("vocab.json ecrit.")


## 5. Tokenizer, feature extractor et processeur

- le **`Wav2Vec2CTCTokenizer`** convertit texte ↔ identifiants de caractères, avec
  `target_lang="ewe"` pour pointer sur notre vocabulaire ;
- le **`Wav2Vec2FeatureExtractor`** normalise la forme d'onde brute (pas de spectrogramme
  ici, MMS lit le signal directement) ;
- le **`Wav2Vec2Processor`** regroupe les deux, comme le `WhisperProcessor`.

In [ ]:
tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
    target_lang=TARGET_LANG,
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=SAMPLING_RATE,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained(OUTPUT_DIR)
print("Processeur pret. Vocab size :", len(processor.tokenizer))


## 6. Préparation des données

Pour chaque exemple :
- `input_values` = la **forme d'onde** normalisée (et non un spectrogramme) ;
- `labels` = la phrase encodée en identifiants de caractères.

On filtre les audios trop longs pour limiter la VRAM.

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"]) / SAMPLING_RATE
    batch["labels"] = processor(text=batch["sentence"]).input_ids
    return batch

ds_prep = ds.map(
    prepare_dataset,
    remove_columns=ds["train"].column_names,
    desc="Preparation (forme d'onde + labels)",
)

ds_prep = ds_prep.filter(lambda l: l < MAX_AUDIO_SEC, input_columns=["input_length"])
print(ds_prep)


## 7. Le *data collator* CTC (padding dynamique)

Formes d'onde et transcriptions ont des longueurs variables. Le collator complète
(*padding*) séparément :
- les `input_values` (audio) via le `feature_extractor` ;
- les `labels` via le `tokenizer`, en mettant les positions de padding à **-100** pour
  qu'elles soient **ignorées** par la perte CTC.

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        # padding de l'audio
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        # padding des labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, padding=self.padding, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)


## 8. Métriques : WER et CER

Pour un modèle CTC, la prédiction est l'**argmax** des logits à chaque pas de temps ;
`batch_decode` applique la fusion CTC (suppression des répétitions et des blancs).
Le **WER** (mots) et le **CER** (caractères) mesurent le taux d'erreur — **plus bas = mieux**.

In [ ]:
metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    # remettre le pad_token a la place des -100 pour decoder les references
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    wer = 100 * metric_wer.compute(predictions=pred_str, references=label_str)
    cer = 100 * metric_cer.compute(predictions=pred_str, references=label_str)
    return {"wer": wer, "cer": cer}


## 9. Chargement du modèle + adaptateur éwé

C'est le cœur de l'approche MMS :

1. on charge MMS-1B avec **notre** taille de vocabulaire (`ignore_mismatched_sizes=True`
   car la tête de sortie change de dimension) ;
2. `init_adapter_layers()` (ré)initialise les couches d'adaptateur ;
3. `freeze_base_model()` **gèle le milliard de paramètres** partagés ;
4. on **réactive uniquement** les poids de l'adaptateur → on n'entraîne que ~2 M paramètres.

Résultat : entraînement rapide, peu gourmand en VRAM, idéal pour un T4.

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    layerdrop=0.0,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,   # essentiel : met a 0 les pertes CTC infinies (texte plus long que l'audio) -> evite NaN
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,
)

# (Re)initialiser puis n'entrainer QUE l'adaptateur de langue
model.init_adapter_layers()
model.freeze_base_model()

adapter_weights = model._get_adapters()
for param in adapter_weights.values():
    param.requires_grad = True

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Parametres entraines : {n_train:,} / {n_total:,} ({100*n_train/n_total:.3f} %)")

## 10. Entraînement

On utilise un `Trainer` **standard** (pas `Seq2SeqTrainer` : CTC ne « génère » pas, il
classe chaque pas de temps). Réglages adaptés au T4, avec reprise automatique sur le dernier
checkpoint pour les runs Kaggle en arrière-plan.

In [ ]:
training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 8,        # batch effectif = 32
    per_device_eval_batch_size  = 8,
    learning_rate = 3e-4,                   # abaisse : 1e-3 faisait diverger l'adaptateur (collapse CTC)
    warmup_steps  = 50 if TRIAL else 500,   # warmup plus long en run complet (stabilite)
    max_steps     = 600 if TRIAL else 8000, # essai court vs ~6.8 epoques sur 37 k clips
    gradient_checkpointing = True,
    fp16 = (device == "cuda"),
    eval_strategy = "steps",
    eval_steps    = 100 if TRIAL else 500,
    save_steps    = 100 if TRIAL else 500,
    logging_steps = 25,
    load_best_model_at_end = True,
    metric_for_best_model  = "wer",
    greater_is_better      = False,         # WER : plus bas = mieux
    save_total_limit       = 2,
    disable_tqdm           = True,
    report_to              = REPORT_TO,     # W&B si cle presente, sinon TensorBoard
    run_name               = "mms-ewe-mixed-trial" if TRIAL else "mms-ewe-mixed",
    # Securite : checkpoints Hub pour reprise multi-session (DESACTIVES pendant l'essai).
    push_to_hub      = (not TRIAL) and bool(HF_TOKEN_WRITE),
    hub_model_id     = PUSH_REPO,
    hub_strategy     = "checkpoint",        # pousse "last-checkpoint" a chaque save
    hub_private_repo = True,
    hub_token        = HF_TOKEN_WRITE,
)

trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = ds_prep["train"],
    eval_dataset     = ds_prep["validation"],
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    processing_class = processor,
)
print("Trainer MMS pret.")

In [ ]:
# Baseline avant entraînement (sur validation)
baseline_metrics = trainer.evaluate(ds_prep["validation"], metric_key_prefix="baseline")

baseline_summary = {
    "split": "validation",
    "wer": float(baseline_metrics.get("baseline_wer")),
    "cer": float(baseline_metrics.get("baseline_cer")),
    "loss": float(baseline_metrics.get("baseline_loss")) if baseline_metrics.get("baseline_loss") is not None else None,
}

baseline_path = Path(OUTPUT_DIR) / "baseline_metrics.json"
with open(baseline_path, "w", encoding="utf-8") as f:
    json.dump(baseline_summary, f, ensure_ascii=False, indent=2)

print("=== BASELINE (avant entraînement) ===")
print(f"WER : {baseline_summary['wer']:.2f} %")
print(f"CER : {baseline_summary['cer']:.2f} %")
print(f"Sauvegardé dans : {baseline_path}")

In [ ]:
from huggingface_hub import snapshot_download

last_ckpt = None
output_path = Path(OUTPUT_DIR)

# 1) Checkpoint local (reprise au sein de la meme session)
if output_path.is_dir():
    ckpts = sorted(
        [d for d in output_path.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[-1]),
    )
    if ckpts:
        last_ckpt = str(ckpts[-1])
        print(f"Reprise locale : {last_ckpt}")

# 2) Sinon, checkpoint pousse sur le Hub (reprise entre sessions Kaggle / apres coupure).
#    Ignore pendant l'essai (TRIAL) pour repartir de zero, sans toucher au depot du run principal.
if last_ckpt is None and HF_TOKEN_WRITE and not TRIAL:
    try:
        snapshot_download(repo_id=PUSH_REPO, allow_patterns="last-checkpoint/*",
                          local_dir=OUTPUT_DIR, token=HF_TOKEN_WRITE)
        cand = output_path / "last-checkpoint"
        if (cand / "trainer_state.json").exists():
            last_ckpt = str(cand)
            print(f"Reprise Hub : {last_ckpt}")
    except Exception as e:
        print(f"Aucun checkpoint Hub ({type(e).__name__}: {e})")

print("Reprise depuis :", last_ckpt if last_ckpt else "from scratch")
train_result = trainer.train(resume_from_checkpoint=last_ckpt)

# Sauvegarde : seul l'adaptateur de langue est specifique a l'eve
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
model.save_pretrained(OUTPUT_DIR)   # ecrit adapter.<lang>.safetensors
print(f"\nModele MMS sauvegarde : {OUTPUT_DIR}")
print(f"Loss train finale : {train_result.training_loss:.4f}")

## 11. Évaluation finale + transcription d'un exemple

In [ ]:
# Évaluation après entraînement
final_val = trainer.evaluate(ds_prep["validation"], metric_key_prefix="final_val")
final_test = trainer.evaluate(ds_prep["test"], metric_key_prefix="final_test")

baseline_path = Path(OUTPUT_DIR) / "baseline_metrics.json"
baseline_summary = None
if baseline_path.exists():
    with open(baseline_path, "r", encoding="utf-8") as f:
        baseline_summary = json.load(f)

print("=== APRÈS ENTRAÎNEMENT ===")
print(f"Validation WER : {final_val['final_val_wer']:.2f} %")
print(f"Validation CER : {final_val['final_val_cer']:.2f} %")
print(f"Test WER       : {final_test['final_test_wer']:.2f} %")
print(f"Test CER       : {final_test['final_test_cer']:.2f} %")

report = {
    "baseline_validation": baseline_summary,
    "final_validation": {
        "wer": float(final_val["final_val_wer"]),
        "cer": float(final_val["final_val_cer"]),
        "loss": float(final_val["final_val_loss"]) if final_val.get("final_val_loss") is not None else None,
    },
    "final_test": {
        "wer": float(final_test["final_test_wer"]),
        "cer": float(final_test["final_test_cer"]),
        "loss": float(final_test["final_test_loss"]) if final_test.get("final_test_loss") is not None else None,
    },
}

if baseline_summary is not None:
    delta_wer = report["final_validation"]["wer"] - baseline_summary["wer"]
    delta_cer = report["final_validation"]["cer"] - baseline_summary["cer"]
    report["delta_validation"] = {"wer": float(delta_wer), "cer": float(delta_cer)}

    print("=== COMPARAISON (validation) ===")
    print(f"Delta WER : {delta_wer:+.2f} %")
    print(f"Delta CER : {delta_cer:+.2f} %")

metrics_path = Path(OUTPUT_DIR) / "asr_mms_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"Rapport complet sauvegardé dans : {metrics_path}")

In [ ]:
# Transcrire un exemple audio brut du test set
model.eval().to(device)

def transcrire(audio_array, sr=SAMPLING_RATE):
    inputs = processor(audio_array, sampling_rate=sr, return_tensors="pt")
    with torch.no_grad():
        logits = model(inputs.input_values.to(device)).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return processor.batch_decode(pred_ids)[0]

ex = ds["test"][0]
pred = transcrire(ex["audio"]["array"], ex["audio"]["sampling_rate"])
print("Reference :", ex["sentence"])
print("Predit    :", pred)


## 12. Pour aller plus loin

- **Publier l'adaptateur :** `model.push_to_hub("votre-nom/mms-ewe")` et
  `processor.push_to_hub(...)`. Seul l'adaptateur éwé (~quelques Mo) est spécifique ;
  le gros modèle reste celui de Meta.
- **Réutiliser à l'inférence :** recharger avec
  `Wav2Vec2ForCTC.from_pretrained(OUTPUT_DIR, target_lang="ewe")` puis
  `processor.tokenizer.set_target_lang("ewe")`.
- **Comparer avec Whisper :** le notebook `asr_ewe_whisper.ipynb` reste disponible. MMS est
  généralement meilleur sur l'éwé (langue nativement supportée), Whisper produit parfois une
  ponctuation plus naturelle.
- **Pipeline complet *voix → voix* :** la sortie texte de ce modèle ASR alimente le notebook
  de **traduction** (éwé → français/anglais), puis le notebook **TTS** pour reparler.
